![logo](../images/logo_diive1_128px.png)

# Delete data from database (influxdb)

---
**Notebook version**: `1` (16 Jul 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

# ⚠️ About this notebook — destructive!

This notebook **permanently deletes** data from the InfluxDB database using diive's in-house engine ([`InfluxIO.delete`](../diive/core/io/db/influx/influxio.py), in `diive/core/io/db/influx`, needs `uv sync --group db`). There is no undo.

For safety, **every `dbc.delete(...)` call below is commented out**. Set the parameters for exactly what you want to remove, double-check them, then uncomment the single call you intend to run. Nothing is deleted on a top-to-bottom *Run All*.

Deletion is scoped by `bucket` + `measurements` + `fields` + `data_version` over the `[START, STOP)` time range. A different `data_version` (e.g. `raw`) and a different bucket (e.g. `{SITE}_raw`) are never touched by a delete aimed at processed data.

# ⏱️ Timestamp convention

The database stores timestamps in **UTC**. `START` and `STOP` below are interpreted in the timezone given by `TIMEZONE_OFFSET_TO_UTC_HOURS` (e.g. `1` for CET winter time) and converted to UTC for the delete. `START` **is** included; `STOP` is the upper bound and **is not** included.

## Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

from diive.core.io.db.influx import InfluxIO  # diive's in-house InfluxDB engine (needs: uv sync --group db)

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
print(f"Last run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

# ✏️ Config folder

In [ ]:
DIRCONF = r'F:\dev\poet\configs'  # <-- set to your config folder
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# 🔌 Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional — print the full `delete` docstring (all parameters and targeting examples):

In [ ]:
# help(dbc.delete)

# 🎯 How targeting works

`measurements` and `fields` each accept a **list** or **`True`** (= all):

| Goal | `measurements` | `fields` |
|---|---|---|
| Specific variables in specific measurements | `['TA', 'SW']` | `['TA_T1_1_1', 'SW_T1_1_1']` |
| All variables of a measurement | `['TA']` | `True` |
| Specific variables across all measurements | `True` | `['TA_T1_1_1']` |
| Everything of a data version | `True` | `True` |

Every deletion is additionally scoped to one `data_version` and the `[START, STOP)` range.

## Delete specific variables
Removes only the named `FIELDS` in the named `MEASUREMENTS`, for the given `DATA_VERSION` and time range.

> ⚠️ Destructive. Uncomment the `dbc.delete(...)` call to run it.

In [ ]:
BUCKET = 'ch-tan_processed'
DATA_VERSION = 'meteoscreening_diive'
MEASUREMENTS = ['LW']
FIELDS = ['LW_BC_IN_T1_2_1', 'LW_BC_OUT_T1_2_1']
START = '2021-05-05 00:00:01'  # included
STOP = '2023-11-29 00:00:01'   # not included
TIMEZONE_OFFSET_TO_UTC_HOURS = 1

# Uncomment to permanently delete the variables above:
# dbc.delete(
#     bucket=BUCKET,
#     measurements=MEASUREMENTS,
#     fields=FIELDS,
#     start=START,
#     stop=STOP,
#     timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
#     data_version=DATA_VERSION,
# )

## Delete all data of a specific data version
`MEASUREMENTS = True` and `FIELDS = True` remove **every variable in every measurement** for the given `DATA_VERSION` and time range. Use a wide time range to catch everything.

> ⚠️ Very destructive — this wipes an entire data version. Uncomment the `dbc.delete(...)` call to run it.

In [ ]:
BUCKET = 'ch-aws_processed'
DATA_VERSION = 'fluxnet_ww2020'
MEASUREMENTS = True  # True = all measurements
FIELDS = True        # True = all fields
START = '1995-01-01 00:00:01'  # included
STOP = '2027-01-01 00:00:01'   # not included
TIMEZONE_OFFSET_TO_UTC_HOURS = 1

# Uncomment to permanently delete the ENTIRE data version above:
# dbc.delete(
#     bucket=BUCKET,
#     measurements=MEASUREMENTS,
#     fields=FIELDS,
#     start=START,
#     stop=STOP,
#     timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
#     data_version=DATA_VERSION,
# )

# ✅ End of notebook

In [ ]:
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")